# GroceryCo Audit - 02 - Reorder Cascade Effect (Bullwhip Analogue)

**Task 1, Part B.** Use complex SQL joins to compute the **Reorder Cascade Effect** - how reorder rates change as `days_since_prior_order` (the lead time analogue) increases. This is the GroceryCo equivalent of supply-chain bullwhip analysis.

**Hypothesis:** Short reorder gaps (1-3 days) should show high reorder rates (impulse refills). Long reorder gaps (20-30 days) should show lower reorder rates (planned shopping with new exploration). The shape and slope of this curve, plus its variance per department, is the audit signal.

**Why this is the bullwhip analogue:** in supply chains, demand variance amplifies upstream as lead times lengthen. In retail customer data, reorder rate variance amplifies as reorder gaps lengthen - same concept, different domain.

## Setup - auto-resolving paths

Run this cell first.

In [ ]:
# Auto-resolving path setup
# Folder layout expected:
#   Module 01/
#     ├── Track 01/                                (this audit)
#     │   ├── dataset/                             (5 CSVs from Instacart Kaggle)
#     │   └── grocery_audit/
#     │       ├── data/
#     │       ├── outputs/
#     │       ├── figures/
#     │       └── (notebooks live here)

from pathlib import Path

def find_project_root():
    p = Path.cwd().resolve()
    for parent in [p] + list(p.parents):
        if parent.name == "grocery_audit":
            return parent
        if (parent / "outputs").exists() and (parent / "data").exists() and (parent / "figures").exists():
            return parent
    return Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

PROJECT_ROOT = find_project_root()
DATASET_DIR  = PROJECT_ROOT.parent / "dataset"
DATA_DIR     = PROJECT_ROOT / "data"
OUTPUTS_DIR  = PROJECT_ROOT / "outputs"
FIGURES_DIR  = PROJECT_ROOT / "figures"

for d in [DATA_DIR, OUTPUTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Dataset dir  : {DATASET_DIR}")
print(f"Data dir     : {DATA_DIR}")
print(f"Outputs dir  : {OUTPUTS_DIR}")
print(f"Figures dir  : {FIGURES_DIR}")

# Locate the 5 CSVs - accept dataset/ or data/
EXPECTED = ["aisles.csv", "departments.csv", "products.csv", "orders.csv", "order_products__train.csv"]
csv_paths = {}
for name in EXPECTED:
    for folder in [DATASET_DIR, DATA_DIR]:
        if (folder / name).exists():
            csv_paths[name] = folder / name
            break
missing = [f for f in EXPECTED if f not in csv_paths]
assert not missing, (
    f"Missing CSVs: {missing}. Place them in {DATASET_DIR} or {DATA_DIR}"
)
print(f"Found {len(csv_paths)} CSVs: {list(csv_paths.keys())}")


## Step 1 - Connect to the SQLite database from Notebook 01

In [ ]:
import sqlite3
import pandas as pd
import numpy as np

db_path = OUTPUTS_DIR / "groceryco.db"
assert db_path.exists(), f"Run Notebook 01 first to create {db_path}"

conn = sqlite3.connect(db_path)
print(f"Connected: {db_path}")
print(f"Size: {db_path.stat().st_size / 1024 / 1024:.1f} MB")


## Step 2 - Bucket days_since_prior_order into lead-time bands

We create 6 lead-time buckets from the customer reorder gap. The first order in a user's history has NULL `days_since_prior_order` and is excluded from this analysis.

In [ ]:
sql_buckets = '''
SELECT
    CASE
        WHEN days_since_prior_order BETWEEN 0 AND 3   THEN '01_0-3_days'
        WHEN days_since_prior_order BETWEEN 4 AND 7   THEN '02_4-7_days'
        WHEN days_since_prior_order BETWEEN 8 AND 14  THEN '03_8-14_days'
        WHEN days_since_prior_order BETWEEN 15 AND 21 THEN '04_15-21_days'
        WHEN days_since_prior_order BETWEEN 22 AND 29 THEN '05_22-29_days'
        WHEN days_since_prior_order = 30              THEN '06_30+_days_capped'
        ELSE NULL
    END AS lead_bucket,
    COUNT(*) AS n_orders
FROM orders
WHERE days_since_prior_order IS NOT NULL
GROUP BY lead_bucket
ORDER BY lead_bucket
'''
buckets = pd.read_sql_query(sql_buckets, conn)
print("Lead-time bucket distribution:")
print(buckets.to_string(index=False))


## Step 3 - The Reorder Cascade query (complex 4-table join)

This is the centrepiece SQL for the audit. It joins 4 tables (orders, order_items, products, departments), buckets by lead time, and computes per-bucket per-department reorder rates. The result is a 6 x 21 = 126-row pivot showing how reorder behaviour decays with reorder gap, by category.

In [ ]:
sql_cascade = '''
WITH bucketed AS (
    SELECT
        o.order_id,
        CASE
            WHEN o.days_since_prior_order BETWEEN 0 AND 3   THEN '01_0-3_days'
            WHEN o.days_since_prior_order BETWEEN 4 AND 7   THEN '02_4-7_days'
            WHEN o.days_since_prior_order BETWEEN 8 AND 14  THEN '03_8-14_days'
            WHEN o.days_since_prior_order BETWEEN 15 AND 21 THEN '04_15-21_days'
            WHEN o.days_since_prior_order BETWEEN 22 AND 29 THEN '05_22-29_days'
            WHEN o.days_since_prior_order = 30              THEN '06_30+_days_capped'
        END AS lead_bucket
    FROM orders o
    WHERE o.days_since_prior_order IS NOT NULL
)
SELECT
    b.lead_bucket,
    d.department,
    COUNT(*)                       AS n_items,
    AVG(oi.reordered)              AS reorder_rate,
    AVG(oi.reordered) * 100.0      AS reorder_pct
FROM bucketed b
JOIN order_items oi ON b.order_id = oi.order_id
JOIN products p     ON oi.product_id = p.product_id
JOIN departments d  ON p.department_id = d.department_id
GROUP BY b.lead_bucket, d.department
ORDER BY b.lead_bucket, d.department
'''
cascade = pd.read_sql_query(sql_cascade, conn)
print(f"Cascade table: {len(cascade)} rows (6 buckets x 21 departments)")
print(f"\nFirst 10 rows:")
print(cascade.head(10).to_string(index=False))


## Step 4 - Pivot for visualisation

In [ ]:
pivot = cascade.pivot(index="department", columns="lead_bucket", values="reorder_pct")

# Compute the "cascade slope" per department: reorder_rate_at_short_lead - reorder_rate_at_long_lead
# A LARGE positive slope means demand variance amplifies as lead time grows (bullwhip-like)
short = pivot["01_0-3_days"]
long_ = pivot["05_22-29_days"]
slope = (short - long_).round(2)
slope_df = slope.reset_index().rename(columns={0: "cascade_slope_pp"})
slope_df.columns = ["department", "cascade_slope_pp"]
slope_df = slope_df.sort_values("cascade_slope_pp", ascending=False)
print("Cascade slope per department (pp = percentage points):")
print("(short-lead reorder rate MINUS long-lead reorder rate)")
print(slope_df.to_string(index=False))

slope_df.to_csv(OUTPUTS_DIR / "cascade_slope_by_department.csv", index=False)


## Step 5 - Visualise the cascade

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

# Heatmap: department (y) x lead_bucket (x) coloured by reorder %
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="RdYlGn",
            center=60, vmin=20, vmax=80,
            cbar_kws={"label": "Reorder rate (%)"},
            linewidths=0.4, linecolor="white", ax=ax)
ax.set_title("Reorder Cascade Effect - reorder rate by lead-time bucket and department",
             fontweight="bold", pad=12)
ax.set_xlabel("Lead-time bucket (days since prior order)")
ax.set_ylabel("Department")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "cascade_heatmap.png", dpi=160, bbox_inches="tight")
plt.show()


### Line chart - cascade per department (top 5 by total volume)

In [ ]:
# Get top 5 departments by total volume
vol = cascade.groupby("department")["n_items"].sum().sort_values(ascending=False)
top5 = vol.head(5).index.tolist()
print(f"Top 5 by volume: {top5}")

fig, ax = plt.subplots(figsize=(11, 6))
order_x = ["01_0-3_days","02_4-7_days","03_8-14_days","04_15-21_days","05_22-29_days","06_30+_days_capped"]
labels_x = ["0-3","4-7","8-14","15-21","22-29","30+(capped)"]

palette = sns.color_palette("rocket", n_colors=5)
for i, dept in enumerate(top5):
    sub = cascade[cascade["department"] == dept].set_index("lead_bucket").reindex(order_x)
    ax.plot(labels_x, sub["reorder_pct"].values, marker="o", linewidth=2.3,
            label=dept, color=palette[i])

ax.set_xlabel("Lead-time bucket (days since prior order)")
ax.set_ylabel("Reorder rate (%)")
ax.set_title("Reorder Cascade - top 5 departments by volume",
             fontweight="bold", pad=10)
ax.legend(loc="best", fontsize=10)
ax.set_ylim(0, 80)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "cascade_top5_lines.png", dpi=160, bbox_inches="tight")
plt.show()


## Step 6 - Variance amplification (the actual bullwhip metric)

Bullwhip effect = ratio of demand variance at upstream stage to downstream stage. Our analogue: ratio of variance in reorder rates ACROSS departments at long lead times vs short lead times. If variance is HIGHER at long lead times, the system shows bullwhip-like amplification.

In [ ]:
variance_per_bucket = cascade.groupby("lead_bucket")["reorder_pct"].agg(
    ["mean", "std", "var"]
).round(3)
variance_per_bucket["coef_of_var"] = (variance_per_bucket["std"] /
                                       variance_per_bucket["mean"]).round(4)
print("Cross-department variance by lead-time bucket:")
print(variance_per_bucket.to_string())

# Bullwhip ratio
bw = variance_per_bucket.loc["05_22-29_days", "var"] / variance_per_bucket.loc["01_0-3_days", "var"]
print(f"\nVariance ratio (long lead / short lead): {bw:.3f}")
if bw > 1.0:
    print(f"  -> AMPLIFICATION present: variance grows {(bw-1)*100:.1f}% as lead time grows")
    print(f"     This is the bullwhip-analogue signal in customer reorder behaviour.")
else:
    print(f"  -> Variance damped at long leads: stable mature demand")

variance_per_bucket.to_csv(OUTPUTS_DIR / "variance_per_lead_bucket.csv")

conn.close()


**Notebook 02 complete.** Cascade table + variance ratio saved. Move to `03_skewness_spc_control_charts.ipynb`.